In [ ]:
import sys
sys.path.insert(0, "/workspace/src")

import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import zarr
from IPython.display import HTML

from datasets.cwb import get_zarr_dataset, get_zarr_dataset_cut  # noqa: F401

plt.rcParams["animation.embed_limit"] = 100  # MB

DATA_PATH = "/workspace/data/cwa_dataset_storm.zarr"
zarr.consolidate_metadata(DATA_PATH)

# Common parameters
DATASET_KWARGS = dict(
    data_path=DATA_PATH,
    in_channels=[0, 1, 2, 3, 4, 9, 10, 11, 12, 17, 18, 19],
    out_channels=[0, 1, 2, 3],
    img_shape_x=448,
    img_shape_y=448,
    ds_factor=1,  # 1 = full-res WRF target; 4 = degraded target (as in cwb.yaml)
    train=True,
    all_times=True,
)

# --- choose one ---
# dataset = get_zarr_dataset(**DATASET_KWARGS)

dataset = get_zarr_dataset_cut(
    **DATASET_KWARGS,
    snippet_x_offset=300,
    snippet_y_offset=50,
    snippet_length=128,
)

print(f"Dataset type   : {type(dataset).__name__}")
print(f"Dataset length : {len(dataset)}")
print(f"Image shape    : {dataset.image_shape()}")


In [ ]:
# Input (ERA5) and output (CWB) channels
input_channels = dataset.input_channels()
output_channels = dataset.output_channels()

print(f"Input channels  ({len(input_channels)}):")
for ch in input_channels:
    print(f"  {ch.name:<40} level={ch.level}")

print(f"\nOutput channels ({len(output_channels)}):")
for ch in output_channels:
    print(f"  {ch.name:<40} level={ch.level}")


In [ ]:
# Normalization stats for the selected channels
info = dataset.info()
era5_center, era5_scale = info["input_normalization"]
cwb_center, cwb_scale   = info["target_normalization"]

print(f"{'Input variable':<40} {'Center':>10} {'Scale':>10}")
print("-" * 62)
for ch, c, s in zip(input_channels, era5_center[list(dataset.in_channels)], era5_scale[list(dataset.in_channels)]):
    print(f"  {ch.name:<38} {c:10.4f} {s:10.4f}")

print(f"\n{'Output variable':<40} {'Center':>10} {'Scale':>10}")
print("-" * 62)
for ch, c, s in zip(output_channels, cwb_center[list(dataset.out_channels)], cwb_scale[list(dataset.out_channels)]):
    print(f"  {ch.name:<38} {c:10.4f} {s:10.4f}")


In [ ]:
# Sample shape and time range
y, x = dataset[0]
print(f"Input tensor  shape: {tuple(x.shape)}  (channels={x.shape[0]}, H={x.shape[1]}, W={x.shape[2]})")
print(f"Target tensor shape: {tuple(y.shape)}  (channels={y.shape[0]}, H={y.shape[1]}, W={y.shape[2]})")
print(f"\nTime range: {dataset.time()[0]}  →  {dataset.time()[-1]}")


In [ ]:
def _get_array(dataset, source, channel_idx, sample_idx, normalized):
    """Return a 2-D numpy array for the given source/channel/sample."""
    target, input_ = dataset[sample_idx]
    arr = (input_ if source == "era5" else target)[channel_idx].numpy()
    if not normalized:
        # denormalize: expects shape (1, C, H, W), returns same
        full = (input_ if source == "era5" else target).numpy()[None, :]
        fn = dataset.denormalize_input if source == "era5" else dataset.denormalize_output
        arr = fn(full)[0, channel_idx]
    return arr


def plot_variable(dataset, source, channel_idx, sample_idx=0, cmap="RdBu_r", normalized=True):
    """Plot a single channel from 'era5' (input) or 'cwb' (output).
    channel_idx is relative to the selected channel subset.
    Pass normalized=False to display physical units.
    """
    arr = _get_array(dataset, source, channel_idx, sample_idx, normalized)
    channels = dataset.input_channels() if source == "era5" else dataset.output_channels()
    ch = channels[channel_idx]
    label = f"{ch.name} {ch.level}".strip()
    units = "normalized" if normalized else "physical units"

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(arr, cmap=cmap, origin="upper")
    plt.colorbar(im, ax=ax, label=f"{label} ({units})")
    ax.set_title(f"{source}  [{label}]  —  sample {sample_idx}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()
    plt.show()


In [ ]:
sample_idx = 3

In [ ]:
# in_channels=[0,1,2,3,4,9,10,11,12,17,18,19] — original channel 17 is at subset index 9
plot_variable(dataset, "era5", channel_idx=9, sample_idx=sample_idx, normalized=False)


In [ ]:
plot_variable(dataset, "cwb", channel_idx=1, sample_idx=sample_idx, normalized=False)

In [ ]:
def animate_variable(dataset, source, channel_idx, cmap="RdBu_r", interval=100,
                     start=0, n_steps=None):
    """Animate a normalized channel over a range of dataset samples.

    Args:
        source: "era5" or "cwb"
        channel_idx: index into the selected channel subset
        interval: milliseconds between frames
        start: first dataset index to include
        n_steps: number of frames to animate (None = all remaining)
    """
    channels = dataset.input_channels() if source == "era5" else dataset.output_channels()
    ch = channels[channel_idx]
    label = f"{ch.name} {ch.level}".strip()

    end = start + n_steps if n_steps is not None else len(dataset)
    times = dataset.time()[start:end]

    all_frames = np.stack([
        (dataset[i][1] if source == "era5" else dataset[i][0])[channel_idx].numpy()
        for i in range(start, end)
    ])
    vmin, vmax = all_frames.min(), all_frames.max()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(all_frames[0], cmap=cmap, origin="upper", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label=f"{label} (normalized)")
    title = ax.set_title("")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()

    def update(i):
        im.set_data(all_frames[i])
        title.set_text(f"{source}  [{label}]  —  {times[i]}")
        return im, title

    anim = animation.FuncAnimation(
        fig, update, frames=len(all_frames), interval=interval, blit=True
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
animate_variable(dataset, "cwb", channel_idx=1, sta# 
# 
# 
# 
# 
# rt=24*7, n_steps=24*7)

In [ ]:
def animate_wind_speed(dataset, source, cmap="viridis", interval=100,
                       start=0, n_steps=None):
    """Animate wind speed (sqrt(u² + v²)) from eastward/northward_wind_10m.

    Args:
        source: "era5" or "cwb"
        interval: milliseconds between frames
        start: first dataset index to include
        n_steps: number of frames to animate (None = all remaining)
    """
    channels = dataset.input_channels() if source == "era5" else dataset.output_channels()
    names = [ch.name for ch in channels]
    u_idx = names.index("eastward_wind_10m")
    v_idx = names.index("northward_wind_10m")

    end = start + n_steps if n_steps is not None else len(dataset)
    times = dataset.time()[start:end]

    u_frames = np.stack([(dataset[i][1] if source == "era5" else dataset[i][0])[u_idx].numpy() for i in range(start, end)])
    v_frames = np.stack([(dataset[i][1] if source == "era5" else dataset[i][0])[v_idx].numpy() for i in range(start, end)])
    speed_frames = np.sqrt(u_frames ** 2 + v_frames ** 2)
    vmin, vmax = speed_frames.min(), speed_frames.max()

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(speed_frames[0], cmap=cmap, origin="upper", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label="wind speed (normalized)")
    title = ax.set_title("")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.tight_layout()

    def update(i):
        im.set_data(speed_frames[i])
        title.set_text(f"{source}  [wind speed]  —  {times[i]}")
        return im, title

    anim = animation.FuncAnimation(
        fig, update, frames=len(speed_frames), interval=interval, blit=True
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
animate_wind_speed(dataset, "cwb", start=24*7, n_steps=24*7)